# Autocorrelation of a Periodic Spline
We plot in the <span style="color:#c20078">**fuchsia**</span> color the spline $f$ that is autocorrelated (*i.e.*, cross-correlated with itself). The resulting autocorrelation is shown as a <span style="color:#1f77b4">**blue**</span> curve, with data samples at the integers represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import cmath
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline with uniformly distributed coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(
    rng.uniform(low = -1.0, high = 1.0, size = 6),
    degree = 3
)

# Plot
def update_plot (
    normalized = True,
    period = 6,
    degree_f = 3,
    delay_f = 0.0
):
    global f

    # Update of the spline with uniformly distributed coefficients
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.uniform(low = -1.0, high = 1.0, size = period),
            degree = f.degree
        )
    f.degree = degree_f
    f.delay = delay_f

    # Autocorrelation
    r_ff = sk.PeriodicSpline1D.convolve(f.mirrored(), f)
    # Normed autocorrelation
    rho_ff = sk.PeriodicSpline1D.normed_cross_correlate(f, f)

    # Plot of the splines
    subplot = plt.subplots()
    if normalized:
        f.plot(
            subplot,
            plotrange = sk.interval.Closed((-1.05, 1.05)),
            plotpoints = 200 + 1,
            curve_fmt = "#c20078",
            curve_markerfmt = " ",
            curvestem_linefmt = "None",
            knot_marker = " "
        )
        rho_ff.plot(
            subplot,
            plotpoints = 200 + 1,
            knot_marker = " "
        )
    else:
        # Dynamic range
        image = {f.image(), r_ff.image()}
        plotrange = sk.interval.Interval.enclosure(image)
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.55 * plotrange.diameter,
            plotrange.midpoint + 0.55 * plotrange.diameter
        ))
        # Plot
        f.plot(
            subplot,
            plotrange = plotrange,
            plotpoints = 200 + 1,
            curve_fmt = "#c20078",
            curve_markerfmt = " ",
            curvestem_linefmt = "None",
            knot_marker = " "
        )
        r_ff.plot(
            subplot,
            plotpoints = 200 + 1,
            knot_marker = " "
        )
    plt.show()

# Interaction
normalized_checkbox_widget = widgets.Checkbox(
    value = True,
    description = "Normalized autocorrelation"
)
widgets.interactive(
    update_plot,
    normalized = normalized_checkbox_widget,
    period = (2, max_period),
    degree_f = (0, max_degree),
    delay_f = (-max_delay, max_delay)
)
